<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Tıp Fakültesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 4 — Sınıflandırma: "Aktif mi, Değil mi?"

**Girdi:** Oturum 2'nin özellik setleri + Oturum 3'ün eğitim/test bölünmesi
**Çıktı:** eğitilmiş sınıflandırıcı ve sonuç tablosu

Oturum 3'te `pIC50` değerini **sayı olarak** tahmin ettik. Şimdi aynı veriyi
farklı bir soruyla modelliyoruz: **bu molekül aktif mi, değil mi?**

Eşiği Oturum 1'de seçmiştiniz; bu defter o kararı `esik_secimi.json`
dosyasından okur ve etiketleri olduğu gibi kullanır. Yeniden eşik belirlemez.

## Neden ikisini de öğreniyoruz?

| | **Regresyon** | **Sınıflandırma** |
|---|---|---|
| **Çıktı** | `pIC50 = 7.8` | "Aktif", olasılık 0.86 |
| **Ne zaman?** | Potensi sıralamak | Büyük kütüphaneleri hızla elemek |
| **Avantajı** | Daha bilgilendirici — ne kadar potent? | Gürültüye daha dayanıklı |
| **Zayıflığı** | Deneysel gürültüye hassas | Eşiğin iki yanındaki moleküller aynı sayılır |

Sanal taramada tipik akış ikisini birleştirir: **sınıflandırıcı** milyonlarca
molekülü hızla eler, ardından **regresyon** modeli kalanları sıralar. Oturum
6'da tam olarak bunu yapacağız.

## Bu oturumun yeni kavramları

1. **Sınıf dengesizliği.** "Her şeye aktif de" diyen aptal bir model bile
   yüksek *doğruluk* alabilir. Bu tuzağı ve çözümünü göreceğiz.
2. **Metrik seçimi.** Doğruluk neden yetersiz; dengeli doğruluk, MCC, ROC-AUC
   ve PR-AUC ne söyler?
3. **Karar eşiği.** Modelin ürettiği *olasılığı* sınıfa çevirirken kullanılan
   0.5 sayısı bir **tercihtir**. Onu değiştirmek, modeli yeniden eğitmeden
   duyarlılık/kesinlik dengesini kaydırır.

⏱️ Yaklaşık süre: 10–20 dakika

---
## 1. Ayarlar

Oturum 3'ün aynısı, iki eklemeyle:

| Ayar | Ne işe yarar? |
|---|---|
| `OPTUNA_METRIK` | Optuna'nın en iyilemeye çalışacağı metrik. `"roc_auc"` karar eşiğinden bağımsız olduğu için ayarlamada en güvenli tercih; `"matthews_corrcoef"` dengesiz veride çok bilgilendirici. |
| `SINIF_AGIRLIGI` | `"balanced"` yapıldığında modeller azınlık sınıfına daha fazla ağırlık verir. |

> ⏱️ Kısaltmak için `OPTUNA_DENEME = 10` ve `Y_RANDOM_TEKRAR = 5` yapın.

In [ ]:
# ── Hedef ─────────────────────────────────────────────────────────────────────
TARGET_ADI  = "NTRK1"
HEDEF_KOLON = "pIC50"

# Siniflandirma esigi BURADA YOK: Oturum 1'de secilmistir. Bu defter o karari
# 01_veri/esik_secimi.json dosyasindan okur ve etiketleri oldugu gibi kullanir.

# ── Google Drive ANA klasoru (Oturum 1-3 ile AYNI) ───────────────────────────
DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"

# ── Girdi dosyalari ──────────────────────────────────────────────────────────
AD_DESC    = "ntrk1_descriptors.csv"
AD_ECFP    = "ntrk1_ecfp4.csv"
AD_MACCS   = "ntrk1_maccs.csv"
AD_BOLUNME = "ntrk1_egitim_test_bolunmesi.csv"   # Oturum 3'un kaydettigi bolunme

# ── Cikti dosyalari ──────────────────────────────────────────────────────────
AD_SONUC = "ntrk1_siniflandirma_sonuclari.csv"
AD_MODEL = "ntrk1_en_iyi_siniflandirma_modeli.joblib"

# ── Modelleme ayarlari ───────────────────────────────────────────────────────
OZELLIK_SETI    = "descriptors"       # "descriptors" | "ecfp4" | "maccs"
TEST_ORANI      = 0.20
DOGRULAMA_ORANI = 0.25
CV_KAT          = 5
OPTUNA_DENEME   = 20
OPTUNA_METRIK   = "roc_auc"           # veya "matthews_corrcoef"
SINIF_AGIRLIGI  = "balanced"          # veya None
Y_RANDOM_TEKRAR = 10
RASTGELE_TOHUM  = 42

ECFP_TARAMASI = False

print(f"Hedef           : {TARGET_ADI} — Aktiflik (siniflandirma)")
print(f"Ozellik seti    : {OZELLIK_SETI}")
print(f"Optuna metrigi  : {OPTUNA_METRIK}")
print(f"Sinif agirligi  : {SINIF_AGIRLIGI}")

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE, ORTAM = Path(DRIVE_PROJE_KLASORU), "Colab + Drive"
except (ImportError, ModuleNotFoundError):
    PROJE, ORTAM = Path(YEREL_KLASOR), "Yerel"

KLASOR = {
    1: PROJE / "01_veri",
    2: PROJE / "02_ozellikler",
    3: PROJE / "03_regresyon",
    4: PROJE / "04_siniflandirma",
    5: PROJE / "05_analiz",
    6: PROJE / "06_adaylar",
}
for k in KLASOR.values():
    (k / "gorseller").mkdir(parents=True, exist_ok=True)

BU     = KLASOR[4]
GORSEL = BU / "gorseller"

print(f"Ortam       : {ORTAM}")
print(f"Ana klasor  : {PROJE}")
print(f"Bu oturum   : {BU}")

In [ ]:
!pip install lazypredict optuna xgboost -q
print("Kurulum tamam.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (StratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             average_precision_score, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_score,
                             precision_recall_curve, recall_score,
                             roc_auc_score, roc_curve, classification_report)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier,
                              HistGradientBoostingClassifier)
from xgboost import XGBClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 25)

print(f"optuna {optuna.__version__} hazir.")

---
## 2. Özellik setlerini ve etiketleri yüklemek

Oturum 2'nin dosyalarını okuyoruz. Bu kez hedef kolonumuz `Aktiflik`.

Etiketi **yeniden hesaplamıyoruz**: Oturum 1'de verdiğiniz karar dosyada
hazır. Eşiği değiştirmek isterseniz Oturum 1'in "Karar" hücresine dönüp
oradan yeniden çalıştırın — tek bir yerde durması, bütün oturumların aynı
eşikle çalışmasını garanti eder.

Çift eşik seçtiyseniz belirsiz banttaki moleküllerin etiketi boştur; bu defter
onları sessizce atlar ve kaç molekülün atlandığını yazdırır.

In [ ]:
def ozellik_seti_yukle(dosya_adi: str):
    yol = KLASOR[2] / dosya_adi
    if not yol.exists():
        raise FileNotFoundError(
            f"'{dosya_adi}' bulunamadi.\n"
            f"Once Oturum 2'yi (02_ozellik_uretimi.ipynb) ayni Drive "
            f"klasorunu kullanarak calistirin.\nBeklenen yol: {yol}"
        )
    tablo = pd.read_csv(yol)
    ozellikler = [k for k in tablo.columns
                  if k not in ("Smiles", HEDEF_KOLON, "Aktiflik")]
    return tablo[ozellikler], tablo


setler, tablo_ref = {}, None
for ad, dosya in [("descriptors", AD_DESC), ("ecfp4", AD_ECFP), ("maccs", AD_MACCS)]:
    X, tablo = ozellik_seti_yukle(dosya)
    setler[ad] = X
    tablo_ref = tablo
    print(f"  {ad:<14} {X.shape[0]:>5} molekul x {X.shape[1]:>5} ozellik")

# ── Oturum 1'in esik karari ──────────────────────────────────────────────────
yol_esik = KLASOR[1] / "esik_secimi.json"
ESIK_SECIMI = (json.loads(yol_esik.read_text(encoding="utf-8"))
               if yol_esik.exists() else None)

# ── Etiketli satirlari sec ───────────────────────────────────────────────────
# Oturum 1'de CIFT esik secildiyse belirsiz banttaki molekullerin Aktiflik
# degeri bostur. Siniflandirma onlari kullanmaz; regresyon (Oturum 3) kullanir.
etiketli = tablo_ref["Aktiflik"].notna().values
bantta   = int((~etiketli).sum())

setler    = {ad: X[etiketli].reset_index(drop=True) for ad, X in setler.items()}
tablo_ref = tablo_ref[etiketli].reset_index(drop=True)

smiles    = tablo_ref["Smiles"]
y_surekli = tablo_ref[HEDEF_KOLON]
y_sinif   = tablo_ref["Aktiflik"].astype(int)

n_aktif = int(y_sinif.sum())

print()
if ESIK_SECIMI:
    print(f"Oturum 1'de secilen esik   : mod '{ESIK_SECIMI['mod']}'")
    if ESIK_SECIMI["mod"] == "tek":
        ESIK = ESIK_SECIMI["esik"]
        print(f"  Aktif (1)                : {HEDEF_KOLON} >= {ESIK}")
    else:
        ESIK = ESIK_SECIMI["esik"]
        print(f"  Inaktif (0)              : "
              f"{HEDEF_KOLON} <= {ESIK_SECIMI['alt_esik']}")
        print(f"  Aktif (1)                : "
              f"{HEDEF_KOLON} >= {ESIK_SECIMI['ust_esik']}")
        print(f"  Cikarilan belirsiz bant  : "
              f"{ESIK_SECIMI['cikarilan_bant_molekul']} molekul")
else:
    # Esik dosyasi yoksa etiketler yine kullanilir; esigi veriden cikariyoruz.
    ESIK = round(float(y_surekli[y_sinif == 1].min()), 2)
    print(f"esik_secimi.json bulunamadi; etiketlerden cikarilan esik: {ESIK}")

if bantta:
    print(f"\nSiniflandirmaya girmeyen (belirsiz bant): {bantta} molekul")

print(f"\nSinif dagilimi ({len(y_sinif)} molekul)")
print(f"  Aktif (1)   : {n_aktif:>5}  (%{100 * n_aktif / len(y_sinif):.1f})")
print(f"  Inaktif (0) : {len(y_sinif) - n_aktif:>5}  "
      f"(%{100 * (1 - n_aktif / len(y_sinif)):.1f})")
print(f"  Dengesizlik orani: "
      f"{max(n_aktif, len(y_sinif) - n_aktif) / min(n_aktif, len(y_sinif) - n_aktif):.2f} : 1")

---
## 3. Sınıf dengesizliği ve "aptal model" testi

Sınıflandırmada en sık yapılan hata **doğruluk (accuracy) metriğine
güvenmektir**.

Aşağıda bir **temel çizgi** kuruyoruz: hiçbir şey öğrenmeyen, her moleküle
sadece çoğunluk sınıfını atayan bir "model". Bu modelin doğruluğu, gerçek
modelimizin **aşması gereken asgari çizgidir**. Modelimizin doğruluğu bu
sayıya yakınsa, model hiçbir şey öğrenmemiş olabilir.

| Metrik | Ne söyler? | Dengesizlikte güvenilir mi? |
|---|---|---|
| **Doğruluk** | Doğru tahmin oranı | ❌ Hayır — çoğunluk sınıfı şişirir |
| **Dengeli doğruluk** | İki sınıfın duyarlılık ortalaması | ✅ Evet |
| **Duyarlılık** | Gerçek aktiflerin kaçını buldu? | ✅ Evet |
| **Kesinlik** | "Aktif" dediklerinin kaçı gerçekten aktif? | ✅ Evet |
| **F1** | Kesinlik ve duyarlılığın harmonik ortalaması | ✅ Evet |
| **MCC** | Karmaşıklık matrisinin tamamını özetler (−1…+1) | ✅ **En sağlamı** |
| **ROC-AUC** | Sıralama kalitesi, eşikten bağımsız | ⚠️ İyimser olabilir |
| **PR-AUC** | Azınlık sınıfına odaklı sıralama kalitesi | ✅ Dengesizlikte tercih edilir |

> **İlaç keşfi bağlamında:** sanal taramada genellikle **kesinlik** önemlidir —
> sentezlemek pahalıdır, "aktif" dediklerinizin gerçekten aktif olması gerekir.
> Ama bir hedefi kaçırmamak istiyorsanız **duyarlılık** öne çıkar. İkisi
> arasındaki denge, Bölüm 8'deki karar eşiği ayarıyla kurulur.

In [ ]:
cogunluk = int(y_sinif.mode()[0])
temel_dogruluk = max(n_aktif, len(y_sinif) - n_aktif) / len(y_sinif)

print("TEMEL CIZGI — 'her seye cogunluk sinifini soyle' modeli")
print("-" * 62)
print(f"  Tahmin ettigi sinif : {cogunluk} "
      f"({'Aktif' if cogunluk == 1 else 'Inaktif'})")
print(f"  Dogruluk            : {temel_dogruluk:.4f}   <-- ASILMASI GEREKEN CIZGI")
print(f"  Dengeli dogruluk    : 0.5000   (hicbir sey ogrenmedi)")
print(f"  MCC                 : 0.0000   (hicbir sey ogrenmedi)")
print(f"  ROC-AUC             : 0.5000   (rastgele siralama)")
print()
print(f"Modelimiz %{100 * temel_dogruluk:.1f}'in UZERINE cikmali; aksi halde")
print("hicbir sey ogrenmemis demektir. Dengeli dogruluk ve MCC ise bu")
print("tuzaga dusmez: ikisi de temel cizgide sifir cikar.")

---
## 4. Oturum 3 ile **aynı** bölünmeyi kullanmak

Regresyon ve sınıflandırma sonuçlarını Oturum 5'te karşılaştıracağız. Bunun
anlamlı olması için iki modelin **aynı test setinde** değerlendirilmesi gerekir.

Bu yüzden Oturum 3'ün kaydettiği bölünme dosyasını okuyoruz. Eşleştirmeyi
sıraya göre değil **SMILES'a göre** yapıyoruz: çift eşik modunda sınıflandırma
veri seti regresyondan kısa olabilir.

Dosya yoksa aynı rastgele tohumla yeniden üretiyoruz.

> Sınıflandırmada bölünmeyi **katmanlı** yapmak önemlidir: eğitim ve test
> setlerinde sınıf oranları korunur. Aksi hâlde şans eseri test setine çok az
> aktif düşebilir ve metrikler güvenilmez olur.

In [ ]:
idx_hepsi = np.arange(len(y_sinif))
yol_bolunme = KLASOR[3] / AD_BOLUNME
bolunme_bulundu = False

if yol_bolunme.exists():
    # Bolunmeyi SIRAYA gore degil SMILES'a gore esliyoruz: cift esik modunda
    # siniflandirma veri seti regresyondan kisa olabilir.
    tablo_b = (pd.read_csv(yol_bolunme)
               .drop_duplicates(subset="Smiles").set_index("Smiles"))
    rastgele = smiles.map(tablo_b["rastgele_bolum"])
    iskelet  = smiles.map(tablo_b["iskelet_bolum"])

    if rastgele.notna().all():
        egitim_idx   = idx_hepsi[(rastgele == "egitim").values]
        test_idx     = idx_hepsi[(rastgele == "test").values]
        i_egitim_idx = idx_hepsi[(iskelet == "egitim").values]
        i_test_idx   = idx_hepsi[(iskelet == "test").values]
        kaynak = f"Oturum 3'un bolunmesi ({yol_bolunme.name})"
        bolunme_bulundu = True
    else:
        eksik = int(rastgele.isna().sum())
        print(f"!! Bolunme dosyasinda {eksik} molekul bulunamadi; "
              f"bolunme yeniden uretiliyor.")

if not bolunme_bulundu:
    egitim_idx, test_idx = train_test_split(
        idx_hepsi, test_size=TEST_ORANI, random_state=RASTGELE_TOHUM,
        stratify=y_sinif,
    )
    i_egitim_idx = i_test_idx = None
    kaynak = "yeniden uretildi (katmanli, ayni tohum)"

ic_egitim_idx, dogrulama_idx = train_test_split(
    egitim_idx, test_size=DOGRULAMA_ORANI, random_state=RASTGELE_TOHUM,
    stratify=y_sinif.iloc[egitim_idx],
)

print(f"Bolunme kaynagi : {kaynak}")
print(f"  Egitim        : {len(egitim_idx):>5}  "
      f"(aktif %{100 * y_sinif.iloc[egitim_idx].mean():.1f})")
print(f"    ic-egitim   : {len(ic_egitim_idx):>5}  "
      f"(aktif %{100 * y_sinif.iloc[ic_egitim_idx].mean():.1f})")
print(f"    dogrulama   : {len(dogrulama_idx):>5}  "
      f"(aktif %{100 * y_sinif.iloc[dogrulama_idx].mean():.1f})")
print(f"  TEST (tutulu) : {len(test_idx):>5}  "
      f"(aktif %{100 * y_sinif.iloc[test_idx].mean():.1f})")
print("\nSinif oranlari butun bolumlerde benzer olmali.")

---
## 5. LazyClassifier ile hızlı tarama

Oturum 3'teki mantığın aynısı: **doğrulama** seti üzerinde ~25 sınıflandırıcıyı
varsayılan ayarlarıyla deneyip hangi ailelerin umut verdiğini görüyoruz.
Test setine **dokunmuyoruz**.

In [ ]:
from lazypredict.Supervised import LazyClassifier


def tarama_yap(X, y, ic_idx, dog_idx, etiket):
    X_ic,  y_ic  = X.iloc[ic_idx],  y.iloc[ic_idx]
    X_dog, y_dog = X.iloc[dog_idx], y.iloc[dog_idx]

    baslangic = time.time()
    tarayici = LazyClassifier(verbose=0, ignore_warnings=True, predictions=False)
    tablo, _ = tarayici.fit(X_ic, X_dog, y_ic, y_dog)
    sure = time.time() - baslangic

    tablo = tablo.copy()
    tablo["Ozellik seti"] = etiket
    print(f"  {etiket:<14} {len(tablo):>3} model, {sure:>6.1f} s   "
          f"en iyi: {tablo.index[0]} "
          f"(dengeli dogruluk={tablo['Balanced Accuracy'].iloc[0]:.3f})")
    return tablo


print("LazyClassifier taramasi basliyor (dogrulama seti uzerinde)...")
print("-" * 80)

taramalar = [
    tarama_yap(setler["descriptors"], y_sinif, ic_egitim_idx, dogrulama_idx,
               "descriptors"),
    tarama_yap(setler["maccs"], y_sinif, ic_egitim_idx, dogrulama_idx, "maccs"),
]
if ECFP_TARAMASI:
    taramalar.append(tarama_yap(setler["ecfp4"], y_sinif, ic_egitim_idx,
                                dogrulama_idx, "ecfp4"))
else:
    print("  ecfp4          atlandi (ECFP_TARAMASI = False)")

tarama_hepsi = pd.concat(taramalar)
print("\nTarama tamamlandi.")

In [ ]:
tarama = (tarama_hepsi[tarama_hepsi["Ozellik seti"] == OZELLIK_SETI]
          .sort_values("Balanced Accuracy", ascending=False))

print(f"LazyClassifier siralamasi — '{OZELLIK_SETI}' (dogrulama seti)")
print("=" * 86)
gosterilecek = [k for k in ["Balanced Accuracy", "Accuracy", "ROC AUC",
                            "F1 Score", "Time Taken"] if k in tarama.columns]
print(tarama[gosterilecek].head(12).round(4).to_string())
print(f"\nEn iyi model: {tarama.index[0]}")
print(f"Temel cizgi (cogunluk) dogrulugu: {temel_dogruluk:.4f}")

### Bütün modellerin sonuçları — bar grafikleri

Tablo ilk 12 modeli gösterdi; aşağıda **taranan bütün sınıflandırıcılar** var.

- **Solda dengeli doğruluk** — iki sınıfın duyarlılık ortalaması. Kırmızı
  kesikli çizgi `0.5`: hiçbir şey öğrenmeyen bir modelin aldığı değer.
- **Sağda ROC-AUC** — sıralama kalitesi, karar eşiğinden bağımsız.

Yeşil çubuk, Optuna ile ayarlayacağımız modeli gösteriyor.

In [ ]:
metrikler = [("Balanced Accuracy", "Dengeli dogruluk", 0.5),
             ("ROC AUC", "ROC-AUC", 0.5)]
metrikler = [(k, ad, t) for k, ad, t in metrikler if k in tarama.columns]

cizilecek = tarama.sort_values(metrikler[0][0]).fillna(0)
en_iyi_ad = tarama.index[0]
renkler = ["#27ae60" if ad == en_iyi_ad else "#7f8c8d" for ad in cizilecek.index]

fig, eksenler = plt.subplots(1, len(metrikler),
                             figsize=(7 * len(metrikler),
                                      0.3 * len(cizilecek) + 2.2),
                             sharey=True, squeeze=False)
y = np.arange(len(cizilecek))

for eksen, (kolon, ad, temel) in zip(eksenler[0], metrikler):
    eksen.barh(y, cizilecek[kolon], color=renkler, edgecolor="white", height=0.72)
    for i, v in enumerate(cizilecek[kolon]):
        eksen.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=7.5,
                   color="#444")
    eksen.axvline(temel, color="#c0392b", linestyle="--", linewidth=1.5,
                  label=f"sans duzeyi = {temel}")
    eksen.set_xlabel(f"{ad}  (yuksek = iyi)")
    eksen.set_title(f"{ad} — {OZELLIK_SETI} (dogrulama seti)")
    eksen.set_xlim(0, 1.08)
    eksen.legend(loc="lower right", fontsize=8)
    eksen.grid(alpha=0.3, axis="x", linestyle="--")

eksenler[0][0].set_yticks(y, cizilecek.index, fontsize=7.5)
fig.suptitle(f"LazyClassifier — taranan {len(tarama)} modelin tamami",
             fontsize=12, y=1.004)
plt.tight_layout()
plt.savefig(GORSEL / "01_lazypredict_butun_modeller.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"Temel cizgi (cogunluk sinifi) dogrulugu : {temel_dogruluk:.4f}")
print(f"En iyi modelin dogrulugu                : "
      f"{tarama['Accuracy'].iloc[0]:.4f}")
print("\nDikkat: duz dogrulukta fark kucuk gorunur ama DENGELI dogrulukta")
print("ayrim nettir. Temel cizgi modelinin dengeli dogrulugu 0.5'tir.")

---
## 6. Optuna ile hiperparametre optimizasyonu

Oturum 3'teki yapının sınıflandırma karşılığı. İki önemli fark var:

1. **Katmanlı çapraz doğrulama** (`StratifiedKFold`) kullanıyoruz: her katmanda
   sınıf oranları korunur.
2. **Sınıf ağırlığı.** `class_weight="balanced"` veren modellerde azınlık
   sınıfının hataları daha ağır cezalandırılır. XGBoost'ta bunun karşılığı
   `scale_pos_weight` parametresidir; onu sınıf oranından hesaplıyoruz.

In [ ]:
def _pipeline(model, olcekle: bool):
    adimlar = ([("olcekle", StandardScaler())] if olcekle else []) + [("model", model)]
    return Pipeline(adimlar)


# XGBoost icin sinif agirligi karsiligi
oran_negatif_pozitif = float((y_sinif.iloc[egitim_idx] == 0).sum() /
                             max((y_sinif.iloc[egitim_idx] == 1).sum(), 1))


def uzay_random_forest(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=deneme.suggest_int("max_depth", 4, 30),
        min_samples_split=deneme.suggest_int("min_samples_split", 2, 12),
        min_samples_leaf=deneme.suggest_int("min_samples_leaf", 1, 8),
        max_features=deneme.suggest_float("max_features", 0.1, 1.0),
    )
    return _pipeline(RandomForestClassifier(random_state=RASTGELE_TOHUM, n_jobs=-1,
                                        class_weight=SINIF_AGIRLIGI, **p),
                 olcekle=False), p


def uzay_extra_trees(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=deneme.suggest_int("max_depth", 4, 30),
        min_samples_leaf=deneme.suggest_int("min_samples_leaf", 1, 8),
        max_features=deneme.suggest_float("max_features", 0.1, 1.0),
    )
    return _pipeline(ExtraTreesClassifier(random_state=RASTGELE_TOHUM, n_jobs=-1,
                                      class_weight=SINIF_AGIRLIGI, **p),
                 olcekle=False), p


def uzay_xgboost(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 800, step=50),
        max_depth=deneme.suggest_int("max_depth", 2, 12),
        learning_rate=deneme.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=deneme.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=deneme.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda=deneme.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        min_child_weight=deneme.suggest_int("min_child_weight", 1, 10),
    )
    agirlik = oran_negatif_pozitif if SINIF_AGIRLIGI == "balanced" else 1.0
    return _pipeline(XGBClassifier(random_state=RASTGELE_TOHUM, n_jobs=-1,
                               tree_method="hist", eval_metric="logloss",
                               scale_pos_weight=agirlik, **p),
                 olcekle=False), p


def uzay_hist_gb(deneme):
    p = dict(
        max_iter=deneme.suggest_int("max_iter", 100, 600, step=50),
        max_depth=deneme.suggest_int("max_depth", 3, 20),
        learning_rate=deneme.suggest_float("learning_rate", 0.01, 0.3, log=True),
        min_samples_leaf=deneme.suggest_int("min_samples_leaf", 5, 40),
        l2_regularization=deneme.suggest_float("l2_regularization", 1e-3, 10.0,
                                               log=True),
    )
    return _pipeline(HistGradientBoostingClassifier(random_state=RASTGELE_TOHUM,
                                                class_weight=SINIF_AGIRLIGI, **p),
                 olcekle=False), p


def uzay_svc(deneme):
    p = dict(
        C=deneme.suggest_float("C", 0.01, 300, log=True),
        gamma=deneme.suggest_float("gamma", 1e-4, 1.0, log=True),
    )
    return _pipeline(SVC(kernel="rbf", probability=True,
                     class_weight=SINIF_AGIRLIGI,
                     random_state=RASTGELE_TOHUM, **p), olcekle=True), p


def uzay_knn(deneme):
    p = dict(
        n_neighbors=deneme.suggest_int("n_neighbors", 1, 40),
        weights=deneme.suggest_categorical("weights", ["uniform", "distance"]),
        p=deneme.suggest_int("p", 1, 2),
    )
    return _pipeline(KNeighborsClassifier(n_jobs=-1, **p), olcekle=True), p


def uzay_logistic(deneme):
    p = dict(C=deneme.suggest_float("C", 1e-3, 100, log=True))
    return _pipeline(LogisticRegression(max_iter=3000, class_weight=SINIF_AGIRLIGI,
                                    random_state=RASTGELE_TOHUM, **p),
                 olcekle=True), p


def uzay_gradient_boosting(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 500, step=50),
        max_depth=deneme.suggest_int("max_depth", 2, 10),
        learning_rate=deneme.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=deneme.suggest_float("subsample", 0.5, 1.0),
    )
    return _pipeline(GradientBoostingClassifier(random_state=RASTGELE_TOHUM, **p),
                 olcekle=False), p


ARAMA_UZAYLARI = {
    "RandomForestClassifier":         uzay_random_forest,
    "ExtraTreesClassifier":           uzay_extra_trees,
    "XGBClassifier":                  uzay_xgboost,
    "LGBMClassifier":                 uzay_hist_gb,
    "HistGradientBoostingClassifier": uzay_hist_gb,
    "SVC":                            uzay_svc,
    "NuSVC":                          uzay_svc,
    "KNeighborsClassifier":           uzay_knn,
    "LogisticRegression":             uzay_logistic,
    "LogisticRegressionCV":           uzay_logistic,
    "RidgeClassifier":                uzay_logistic,
    "RidgeClassifierCV":              uzay_logistic,
    "BaggingClassifier":              uzay_random_forest,
    "GradientBoostingClassifier":     uzay_gradient_boosting,
    "RandomForestRegressor":          uzay_random_forest,
}

secilen_ad, secilen_uzay = None, None
for model_adi in tarama.index:
    if model_adi in ARAMA_UZAYLARI:
        secilen_ad, secilen_uzay = model_adi, ARAMA_UZAYLARI[model_adi]
        break
if secilen_ad is None:
    secilen_ad, secilen_uzay = "XGBClassifier", uzay_xgboost
    print("Arama uzayi tanimli model bulunamadi; XGBoost'a donuluyor.")

print(f"Optimize edilecek model : {secilen_ad}")
print(f"LazyPredict siralamasi  : {list(tarama.index).index(secilen_ad) + 1}.")

X_tam = setler[OZELLIK_SETI]
X_egitim, y_egitim = X_tam.iloc[egitim_idx], y_sinif.iloc[egitim_idx]
X_test,   y_test   = X_tam.iloc[test_idx],  y_sinif.iloc[test_idx]
print(f"\nEgitim: {X_egitim.shape}   Test: {X_test.shape}")

In [ ]:
kfold = StratifiedKFold(n_splits=CV_KAT, shuffle=True, random_state=RASTGELE_TOHUM)


def amac(deneme):
    pipeline, _ = secilen_uzay(deneme)
    skorlar = cross_val_score(pipeline, X_egitim, y_egitim, cv=kfold,
                              scoring=OPTUNA_METRIK, n_jobs=1)
    return skorlar.mean()


print(f"Optuna: {OPTUNA_DENEME} deneme x {CV_KAT} katli katmanli CV "
      f"({OPTUNA_METRIK})")
print("(birkac dakika surebilir)\n")

baslangic = time.time()
calisma = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RASTGELE_TOHUM),
    study_name=f"{secilen_ad}_siniflandirma",
)
calisma.optimize(amac, n_trials=OPTUNA_DENEME, show_progress_bar=False)
sure = time.time() - baslangic

print(f"Tamamlandi: {sure:.1f} s  ({sure / OPTUNA_DENEME:.1f} s/deneme)")
print(f"\nEn iyi CV {OPTUNA_METRIK}: {calisma.best_value:.4f}")
print("En iyi parametreler:")
for anahtar, deger in calisma.best_params.items():
    print(f"  {anahtar:<22} {deger}")

---
## 7. Son modeli eğitmek ve test setinde değerlendirmek

Test setine ilk ve son dokunuşumuz. Bütün metrikleri birlikte raporluyoruz —
tek bir sayıya bakmak yanıltıcı olur.

In [ ]:
en_iyi_pipeline, _ = secilen_uzay(optuna.trial.FixedTrial(calisma.best_params))

# Capraz dogrulama tahminleri (egitim verisinde, sizintisiz)
cv_tahmin = cross_val_predict(en_iyi_pipeline, X_egitim, y_egitim, cv=kfold, n_jobs=1)
cv_olasilik = cross_val_predict(en_iyi_pipeline, X_egitim, y_egitim, cv=kfold,
                                method="predict_proba", n_jobs=1)[:, 1]

# Son egitim ve test
en_iyi_pipeline.fit(X_egitim, y_egitim)
tahmin_test    = en_iyi_pipeline.predict(X_test)
olasilik_test  = en_iyi_pipeline.predict_proba(X_test)[:, 1]


def siniflama_metrikleri(gercek, tahmin, olasilik=None) -> dict:
    sonuc = {
        "Dogruluk":          accuracy_score(gercek, tahmin),
        "Dengeli dogruluk":  balanced_accuracy_score(gercek, tahmin),
        "Kesinlik":          precision_score(gercek, tahmin, zero_division=0),
        "Duyarlilik":        recall_score(gercek, tahmin, zero_division=0),
        "F1":                f1_score(gercek, tahmin, zero_division=0),
        "MCC":               matthews_corrcoef(gercek, tahmin),
    }
    if olasilik is not None:
        sonuc["ROC-AUC"] = roc_auc_score(gercek, olasilik)
        sonuc["PR-AUC"]  = average_precision_score(gercek, olasilik)
    return sonuc


m_cv   = siniflama_metrikleri(y_egitim, cv_tahmin, cv_olasilik)
m_test = siniflama_metrikleri(y_test, tahmin_test, olasilik_test)

sonuc = pd.DataFrame([m_cv, m_test],
                     index=[f"Capraz dogrulama ({CV_KAT} kat)", "TEST (tutulan)"])

print(f"Model: {secilen_ad}   |   Ozellik seti: {OZELLIK_SETI}")
print("=" * 94)
print(sonuc.round(4).to_string())

print(f"\nTemel cizgi (cogunluk) dogrulugu : {temel_dogruluk:.4f}")
print(f"Modelin test dogrulugu           : {m_test['Dogruluk']:.4f}")
kazanc = m_test["Dogruluk"] - temel_dogruluk
print(f"Temel cizgi uzerine kazanc       : {kazanc:+.4f}")
print(f"\nAma asil bakilmasi gereken:")
print(f"  Dengeli dogruluk : {m_test['Dengeli dogruluk']:.4f}  (temel cizgi 0.50)")
print(f"  MCC              : {m_test['MCC']:.4f}  (temel cizgi 0.00)")

In [ ]:
print("Siniflandirma raporu (test seti)")
print("=" * 62)
print(classification_report(y_test, tahmin_test,
                            target_names=["Inaktif (0)", "Aktif (1)"],
                            digits=3))

### Karmaşıklık matrisi, ROC ve PR eğrileri

Üç grafik birlikte okunur:

- **Karmaşıklık matrisi** — hataların *türünü* gösterir. Yanlış pozitifler
  (inaktifi aktif sanmak) boşa sentez demektir; yanlış negatifler (aktifi
  kaçırmak) fırsat kaybıdır.
- **ROC eğrisi** — bütün karar eşikleri için duyarlılık/yanlış pozitif dengesi.
  Köşegen (0.5) rastgele tahmindir.
- **PR eğrisi** — kesinlik/duyarlılık dengesi. Dengesiz veride ROC'dan daha
  bilgilendiricidir, çünkü çoğunluk sınıfının büyüklüğünden etkilenmez.
  Yatay kesikli çizgi, azınlık sınıf oranıdır (temel çizgi).

In [ ]:
fig, (sol, orta, sag) = plt.subplots(1, 3, figsize=(16, 4.6))

# --- Karmasiklik matrisi ---
km = confusion_matrix(y_test, tahmin_test)
sol.imshow(km, cmap="Blues")
sol.set_xticks([0, 1]); sol.set_xticklabels(["Inaktif", "Aktif"])
sol.set_yticks([0, 1]); sol.set_yticklabels(["Inaktif", "Aktif"])
sol.set_xlabel("Tahmin"); sol.set_ylabel("Gercek")
sol.set_title("Karmasiklik matrisi (test)")
etiketler = [["Dogru negatif", "Yanlis pozitif"],
             ["Yanlis negatif", "Dogru pozitif"]]
for i in range(2):
    for j in range(2):
        sol.text(j, i, f"{etiketler[i][j]}\n{km[i, j]}", ha="center",
                 va="center", fontsize=9,
                 color="white" if km[i, j] > km.max() / 2 else "black")

# --- ROC ---
fpr, tpr, _ = roc_curve(y_test, olasilik_test)
orta.plot(fpr, tpr, linewidth=2, color="#4878a8",
          label=f"ROC-AUC = {m_test['ROC-AUC']:.3f}")
orta.plot([0, 1], [0, 1], "k--", linewidth=1, label="rastgele (0.5)")
orta.set_xlabel("Yanlis pozitif orani (1 - ozulluk)")
orta.set_ylabel("Dogru pozitif orani (duyarlilik)")
orta.set_title("ROC egrisi")
orta.legend(fontsize=9)
orta.grid(alpha=0.3, linestyle="--")

# --- PR ---
kesinlik, duyarlilik, _ = precision_recall_curve(y_test, olasilik_test)
taban = y_test.mean()
sag.plot(duyarlilik, kesinlik, linewidth=2, color="#c0392b",
         label=f"PR-AUC = {m_test['PR-AUC']:.3f}")
sag.axhline(taban, color="k", linestyle="--", linewidth=1,
            label=f"temel cizgi = {taban:.3f}")
sag.set_xlabel("Duyarlilik (recall)")
sag.set_ylabel("Kesinlik (precision)")
sag.set_title("Kesinlik-duyarlilik egrisi")
sag.legend(fontsize=9)
sag.grid(alpha=0.3, linestyle="--")

plt.suptitle(f"{secilen_ad} — {TARGET_ADI} aktiflik siniflandirmasi", y=1.03)
plt.tight_layout()
plt.savefig(GORSEL / "02_siniflandirma_grafikleri.png", dpi=150,
            bbox_inches="tight")
plt.show()

---
## 8. Karar eşiği: modelin dışındaki bir ayar

Modelimiz aslında sınıf değil **olasılık** üretir: "bu molekülün aktif olma
olasılığı 0.73". Bu olasılığı sınıfa çevirmek için bir eşiğe ihtiyaç var ve
`predict()` sessizce **0.5** kullanır.

Ama 0.5 kutsal bir sayı değil, bir **tercihtir**:

- **Eşiği yükseltmek** (ör. 0.8) → daha az molekül "aktif" denir →
  **kesinlik artar**, duyarlılık düşer. *Sentez bütçeniz kısıtlıysa bunu
  istersiniz.*
- **Eşiği düşürmek** (ör. 0.3) → daha çok molekül "aktif" denir →
  **duyarlılık artar**, kesinlik düşer. *Hiçbir aktifi kaçırmak
  istemiyorsanız.*

Aşağıda eşiği taramalı değiştirip metriklerin nasıl değiştiğini görüyoruz ve
F1'i, MCC'yi en büyükleyen eşikleri işaretliyoruz.

> **Önemli.** Eşiği **test setine bakarak** seçmek bir tür sızıntıdır. Doğrusu
> doğrulama seti ya da çapraz doğrulama tahminleri üzerinde seçmektir. Aşağıda
> bunu doğru yapıyoruz: eşiği **çapraz doğrulama olasılıklarından** seçip sonra
> test setinde uyguluyoruz.

In [ ]:
esikler = np.linspace(0.05, 0.95, 91)

# Esigi CAPRAZ DOGRULAMA olasiliklarindan sec (test setine bakmadan)
satirlar = []
for esik_degeri in esikler:
    tahmin_cv = (cv_olasilik >= esik_degeri).astype(int)
    satirlar.append({
        "Esik": esik_degeri,
        "Dengeli dogruluk": balanced_accuracy_score(y_egitim, tahmin_cv),
        "Kesinlik": precision_score(y_egitim, tahmin_cv, zero_division=0),
        "Duyarlilik": recall_score(y_egitim, tahmin_cv, zero_division=0),
        "F1": f1_score(y_egitim, tahmin_cv, zero_division=0),
        "MCC": matthews_corrcoef(y_egitim, tahmin_cv),
    })
df_esik = pd.DataFrame(satirlar)

esik_f1  = float(df_esik.loc[df_esik["F1"].idxmax(), "Esik"])
esik_mcc = float(df_esik.loc[df_esik["MCC"].idxmax(), "Esik"])

fig, eksen = plt.subplots(figsize=(11, 4.6))
for kolon, renk in [("Kesinlik", "#c0392b"), ("Duyarlilik", "#4878a8"),
                    ("F1", "#27ae60"), ("MCC", "#8e44ad")]:
    eksen.plot(df_esik["Esik"], df_esik[kolon], label=kolon, color=renk,
               linewidth=1.8)
eksen.axvline(0.5, color="k", linestyle="--", linewidth=1,
              label="varsayilan 0.5")
eksen.axvline(esik_mcc, color="#8e44ad", linestyle=":", linewidth=1.8,
              label=f"MCC en iyi = {esik_mcc:.2f}")
eksen.set_xlabel("Karar esigi")
eksen.set_ylabel("Metrik degeri")
eksen.set_title("Karar esigi metrikleri nasil degistirir? "
                "(capraz dogrulama tahminleri uzerinde)")
eksen.legend(fontsize=9, ncol=3)
eksen.grid(alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

# Secilen esigi TEST setine uygula
tahmin_test_ayarli = (olasilik_test >= esik_mcc).astype(int)
m_test_ayarli = siniflama_metrikleri(y_test, tahmin_test_ayarli, olasilik_test)

karsilastirma_esik = pd.DataFrame(
    [m_test, m_test_ayarli],
    index=["Varsayilan esik (0.50)", f"Ayarli esik ({esik_mcc:.2f})"],
)
print("Test setinde esik karsilastirmasi")
print("=" * 94)
print(karsilastirma_esik.round(4).to_string())
print("\nEsik CAPRAZ DOGRULAMA verisinden secildi, test setinden degil.")
print("Bu yuzden yukaridaki test skorlari hala durusttur.")

---
## 9. y-randomizasyon (sınıflandırma)

Regresyonda yaptığımız testin aynısı: etiketleri karıştırıp modeli yeniden
eğitiyoruz. Sınıflandırmada beklenti şudur:

- **ROC-AUC ≈ 0.5** (rastgele sıralama)
- **MCC ≈ 0** (hiçbir ilişki yok)
- **Dengeli doğruluk ≈ 0.5**

Gerçek modelimiz bu sans düzeyinden belirgin biçimde uzak olmalı.

In [ ]:
print(f"y-randomizasyon: {Y_RANDOM_TEKRAR} tekrar\n")

rastgele = np.random.RandomState(RASTGELE_TOHUM)
sahte_auc, sahte_mcc = [], []

baslangic = time.time()
for tekrar in range(Y_RANDOM_TEKRAR):
    y_karisik = pd.Series(rastgele.permutation(y_egitim.values),
                          index=y_egitim.index)
    sahte_pipeline, _ = secilen_uzay(optuna.trial.FixedTrial(calisma.best_params))

    auc = cross_val_score(sahte_pipeline, X_egitim, y_karisik, cv=kfold,
                          scoring="roc_auc", n_jobs=1).mean()
    sahte_auc.append(auc)

    sahte_pipeline.fit(X_egitim, y_karisik)
    sahte_mcc.append(matthews_corrcoef(y_test, sahte_pipeline.predict(X_test)))

    if (tekrar + 1) % 5 == 0:
        print(f"  {tekrar + 1:>2}/{Y_RANDOM_TEKRAR} "
              f"({time.time() - baslangic:.0f} s)")

sahte_auc = np.array(sahte_auc)
sahte_mcc = np.array(sahte_mcc)

print(f"\n{'':30}{'CV ROC-AUC':>13}{'Test MCC':>12}")
print("-" * 56)
print(f"{'GERCEK model':<30}{m_cv['ROC-AUC']:>13.4f}{m_test['MCC']:>12.4f}")
print(f"{'Karistirilmis (ortalama)':<30}{sahte_auc.mean():>13.4f}"
      f"{sahte_mcc.mean():>12.4f}")
print(f"{'Karistirilmis (en iyi)':<30}{sahte_auc.max():>13.4f}"
      f"{sahte_mcc.max():>12.4f}")

ucurum = m_cv["ROC-AUC"] - sahte_auc.max()
print(f"\nROC-AUC ucurumu: {ucurum:.4f}")
if ucurum > 0.15:
    print("  -> BASARILI. Model gercek bir yapi-aktiflik iliskisi ogrenmis.")
else:
    print("  -> DIKKAT! Fark kucuk; sans korelasyonu olabilir.")

In [ ]:
fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.2))

sol.hist(sahte_auc, bins=14, color="#95a5a6", edgecolor="white", alpha=0.85,
         label=f"Karistirilmis (n={Y_RANDOM_TEKRAR})")
sol.axvline(m_cv["ROC-AUC"], color="#c0392b", linewidth=2.5,
            label=f"GERCEK = {m_cv['ROC-AUC']:.3f}")
sol.axvline(0.5, color="k", linestyle="--", linewidth=1, label="sans = 0.5")
sol.set_xlabel("CV ROC-AUC"); sol.set_ylabel("Tekrar")
sol.set_title("y-randomizasyon — ROC-AUC")
sol.legend(fontsize=8); sol.grid(alpha=0.3, axis="y", linestyle="--")

sag.hist(sahte_mcc, bins=14, color="#95a5a6", edgecolor="white", alpha=0.85,
         label="Karistirilmis")
sag.axvline(m_test["MCC"], color="#c0392b", linewidth=2.5,
            label=f"GERCEK = {m_test['MCC']:.3f}")
sag.axvline(0, color="k", linestyle="--", linewidth=1, label="sans = 0")
sag.set_xlabel("Test MCC"); sag.set_ylabel("Tekrar")
sag.set_title("y-randomizasyon — MCC")
sag.legend(fontsize=8); sag.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.savefig(GORSEL / "03_y_randomizasyon.png", dpi=150,
            bbox_inches="tight")
plt.show()

---
## 10. Sonuçları ve modeli kaydetme

In [ ]:
df_sonuc = pd.DataFrame([{
    "Gorev": "siniflandirma",
    "Model": secilen_ad,
    "Ozellik seti": OZELLIK_SETI,
    "Ozellik sayisi": X_tam.shape[1],
    "Esik (pIC50)": ESIK,
    "Karar esigi": 0.5,
    "Egitim n": len(egitim_idx),
    "Test n": len(test_idx),
    "Temel cizgi dogruluk": round(temel_dogruluk, 4),
    **{f"CV {k}": round(v, 4) for k, v in m_cv.items()},
    **{f"Test {k}": round(v, 4) for k, v in m_test.items()},
    "y-random CV ROC-AUC (ort)": round(float(sahte_auc.mean()), 4),
    "y-random CV ROC-AUC (max)": round(float(sahte_auc.max()), 4),
}, {
    "Gorev": "siniflandirma",
    "Model": secilen_ad,
    "Ozellik seti": OZELLIK_SETI,
    "Ozellik sayisi": X_tam.shape[1],
    "Esik (pIC50)": ESIK,
    "Karar esigi": round(esik_mcc, 3),
    "Egitim n": len(egitim_idx),
    "Test n": len(test_idx),
    "Temel cizgi dogruluk": round(temel_dogruluk, 4),
    **{f"CV {k}": None for k in m_cv},
    **{f"Test {k}": round(v, 4) for k, v in m_test_ayarli.items()},
    "y-random CV ROC-AUC (ort)": None,
    "y-random CV ROC-AUC (max)": None,
}])

df_sonuc.to_csv(BU / AD_SONUC, index=False)
tarama_hepsi.to_csv(BU / "ntrk1_siniflandirma_lazypredict_tarama.csv")

paket = {
    "pipeline": en_iyi_pipeline,
    "model_adi": secilen_ad,
    "gorev": "siniflandirma",
    "ozellik_seti": OZELLIK_SETI,
    "ozellik_adlari": list(X_tam.columns),
    "aktiflik_esigi": ESIK,
    "karar_esigi": float(esik_mcc),
    "en_iyi_parametreler": calisma.best_params,
    "cv_roc_auc": m_cv["ROC-AUC"],
    "test_roc_auc": m_test["ROC-AUC"],
    "test_mcc": m_test["MCC"],
    "egitim_indeksleri": egitim_idx.tolist(),
    "test_indeksleri": test_idx.tolist(),
}
joblib.dump(paket, BU / AD_MODEL)

pd.DataFrame({
    "Smiles": smiles.iloc[test_idx].values,
    HEDEF_KOLON: y_surekli.iloc[test_idx].values,
    "gercek_aktiflik": y_test.values,
    "tahmin_aktiflik": tahmin_test,
    "aktif_olasiligi": olasilik_test,
}).to_csv(BU / "ntrk1_siniflandirma_test_tahminleri.csv", index=False)

print("Kaydedilen dosyalar")
print("-" * 70)
for ad in [AD_SONUC, "ntrk1_siniflandirma_lazypredict_tarama.csv", AD_MODEL,
           "ntrk1_siniflandirma_test_tahminleri.csv"]:
    yol = BU / ad
    if yol.exists():
        print(f"  {ad:<52} {yol.stat().st_size / 1024:>8.1f} KB")
print(f"\nKlasor: {BU}")

---
## 11. Özet

### Ne yaptık?

1. Aynı veriyi **ikili sınıflandırma** olarak modelledik — etiketi Oturum 1'in
   eşik kararından okuduk.
2. Bir **temel çizgi** kurup doğruluk metriğinin neden yanıltıcı olduğunu
   gösterdik.
3. Oturum 3 ile **aynı bölünmeyi** kullandık; sonuçlar karşılaştırılabilir.
4. **LazyClassifier** ile tarama, **Optuna** ile ayarlama, **katmanlı 5 katlı
   CV** ile doğrulama yaptık.
5. Karmaşıklık matrisi, ROC ve PR eğrilerini birlikte yorumladık.
6. **Karar eşiğini** çapraz doğrulama verisinden seçip test setinde uyguladık.
7. **y-randomizasyon** ile modeli doğruladık.

### Akılda kalması gerekenler

- **Doğruluğa tek başına asla güvenmeyin.** Her zaman temel çizgiyi birlikte
  bildirin. Dengeli doğruluk ve MCC bu tuzağa düşmez.
- **Dengesiz veride PR-AUC, ROC-AUC'dan daha bilgilendiricidir.**
- **Karar eşiği bir tasarım tercihidir.** Sentez bütçesi kısıtlıysa kesinliği,
  hiçbir aktifi kaçırmamak istiyorsanız duyarlılığı öne alın.
- **Eşiği test setinden seçmeyin** — doğrulama ya da CV verisinden seçin.
- **Sınıf ağırlığı**, dengesizliği ele almanın en basit ve genellikle en
  etkili yoludur.

### Sırada ne var?

**Oturum 5 — Model karşılaştırma, SHAP ve uygulanabilirlik alanı.** Üç oturumun
sonuçlarını tek tabloda toplayacak, **SHAP** ile modelin *neden* öyle tahmin
ettiğini açıklayacak ve **Williams grafiği** ile modelin hangi moleküller için
güvenilir olduğunu belirleyeceğiz.

---

### Alıştırmalar

1. Oturum 1'e dönüp eşiği 6.0 yapın (sınıflar çok dengesizleşir) ve bu defteri
   yeniden çalıştırın. Hangi metrik eşik değişiminden en az etkileniyor?
2. Oturum 1'de **çift eşik** seçin (`ESIK_MODU = "cift"`, `BANT = 0.5`).
   Test metrikleri yükseliyor mu? Neden — model mi iyileşti, yoksa soru mu
   kolaylaştı?
3. `SINIF_AGIRLIGI = None` yapıp karşılaştırın. Dengeli doğruluk ve duyarlılık
   ne oluyor?
4. `OPTUNA_METRIK`'i `"matthews_corrcoef"` yapın. Seçilen hiperparametreler ve
   test metrikleri değişiyor mu?
5. Karar eşiğini kesinliği en büyükleyecek biçimde seçin. Kaç molekülü "aktif"
   diyor, kaçını kaçırıyor? Bir sanal tarama kampanyasında bu tercih hangi
   koşulda doğru olur?